In [1]:
import xarray as xr
import numpy as np
import pandas as pd
import netCDF4 as nc
import os
from datetime import datetime, timedelta
import glob
import tqdm

In [2]:
# Base directories
gruan_base_dir = '/home/chinahg/GCresearch/GRUAN_sondes/'
era5_base_dir = '/home/chinahg/GCresearch/ERA5_downloads/'

# Function to construct ERA5 file path from GRUAN file path
def construct_era5_path(gruan_file_path):
    date_str = os.path.basename(gruan_file_path).split('_')[4][:8]
    era5_file_path = os.path.join(era5_base_dir, date_str[:4], f'{date_str[:4]}_{date_str[4:6]}_{date_str[6:8]}.nc')
    return era5_file_path

# Arrays to store the paths
gruan_file_paths = []
era5_file_paths = []

# Recursively find all GRUAN files and construct corresponding ERA5 file paths
for root, dirs, files in os.walk(gruan_base_dir):
    for file in files:
        if file.endswith('.nc'):
            gruan_file_path = os.path.join(root, file)
            era5_file_path = construct_era5_path(gruan_file_path)
            gruan_file_paths.append(gruan_file_path)
            era5_file_paths.append(era5_file_path)

# Sort the paths
gruan_file_paths.sort()
era5_file_paths.sort()

In [3]:
####################################################################################################
### Functions ###

def fill_nan_with_next(arr):
    """Fill NaN values in the array with the next non-NaN value, if available."""
    for i in range(len(arr) - 1):
        if np.isnan(arr[i]):
            next_valid = next((x for x in arr[i + 1:] if not np.isnan(x)), np.nan)
            arr[i] = next_valid
    return arr

def press2alt(pressure):
    """
    Convert pressure to altitude.

    Parameters
    ----------
    pressure : Union[int, np.ndarray]
        Pressure in Pascal.

    Returns
    -------
    Union[float, np.ndarray]
        Altitude in meters.
    """
    L = -6.5*10**-3
    P0 = 101325
    T0 = 288.15
    R = 287.053
    g = 9.81

    altitudes = np.zeros_like(pressure)

    if type(pressure)==int:
        return (T0/L)*((pressure*100/P0)**(-R*L/g) -1)
    else:
        for i in range(len(pressure)):
            altitudes[i] = (T0/L)*((pressure[i]*100/P0)**(-R*L/g) -1)

        return altitudes

####################################################################################################
valid_combinations = []
G_datetime_data = []
G_site_data = []
G_lat_data = []
G_lon_data = []
G_alt_data = []
G_RHi_data = []
E_alt_data = []
E_RHi_data = []
E_datetime_data = []

# Define the dimensions
days = pd.date_range('2005-01-01', '2021-12-31')  # From 2005 to the end of 2021
E_latitudes = np.linspace(30, 60, int((60 - 30) / 0.25) + 1, 2)  # 0.25 degree increments between 30 and 60 degrees
E_longitudes = np.linspace(-180, 180, int(360 / 0.25) + 1, 2)  # 0.25 degree increments

for j in tqdm.tqdm(len(gruan_file_paths)):
    # Open the ERA5 file
    E_file_path = era5_file_paths[j]
    E_data = xr.open_dataset(E_file_path)

    # Open the GRUAN file
    G_file_path = gruan_file_paths[j]
    G_data = nc.Dataset(G_file_path)

    # Extract the base time from the G_data attributes
    base_time_str = G_data.variables['time'].units.split('since ')[1]
    base_time = datetime.strptime(base_time_str, '%Y-%m-%dT%H:%M:%S')

    G_datetime_data.append([base_time + timedelta(seconds=float(sec)) for sec in G_data.variables['time'][:]]) # Convert the time variable from seconds since base_time to datetime objects

    G_site_data.append(os.path.basename(G_file_path).split('_')[0].split('-')[0])

    G_lat_data.append(fill_nan_with_next(G_data.variables['lat'][:]))
    
    G_lon_data.append(fill_nan_with_next(G_data.variables['lon'][:]))

    G_alt_data.append(fill_nan_with_next(G_data.variables['alt'][:]))

    G_RHi_data.append(G_data.variables['rh_i'][:])

    E_alt_data.append(press2alt(E_data.sel(latitude=G_lat_data[j][0], longitude=G_lon_data[j][0], method='nearest')['isobaricInhPa'])) # use lat_index and lon_index to find the corresponding latitude and longitude, results will be in alt and time dimensions

    E_RHi_data.append(E_data.sel(latitude=G_lat_data[j][0], longitude=G_lon_data[j][0], method='nearest')['RH_i'])

    E_datetime_data.append(E_data.variables['time'][:])

    # Store the data in a dictionary, with variable names as keys
    data_dict = {
        'G_site': G_site_data,
        'G_lat': G_lat_data,
        'G_lon': G_lon_data,
        'G_alt': G_alt_data,
        'G_RHi': G_RHi_data,
        'G_dt': G_datetime_data,
        'E_alt': E_alt_data,
        'E_RHi': E_RHi_data,
        'E_dt': E_datetime_data
    }

    # Create a list of non-empty (day, lat, lon) combinations
    E_latitude = np.float(E_data.latitude.sel(latitude=G_lat_data[j][0], method='nearest').values)
    E_longitude = np.float(E_data.longitude.sel(longitude=G_lon_data[j][0], method='nearest').values)

    combo = (G_datetime_data[j][0].strftime('%Y-%m-%d'), E_latitude, E_longitude)

    valid_combinations += [combo]

# Convert valid combinations into a MultiIndex
index = pd.MultiIndex.from_tuples(valid_combinations, names=['day', 'E_latitude', 'E_longitude'])

# Create an empty list to store the flattened data
flattened_data = []

# Iterate over the MultiIndex to map each (day, lat, lon) combination to its corresponding data
i = 0 # Counter to keep track of the current row
for (day, lat, lon) in index:
    
    # Create the row by selecting data from each variable using the correct indices
    row = [
        data_dict['G_site'][i],
        data_dict['G_lat'][i],
        data_dict['G_lon'][i],
        data_dict['G_alt'][i],
        data_dict['G_RHi'][i],
        data_dict['G_dt'][i],
        data_dict['E_alt'][i],
        data_dict['E_RHi'][i],
        data_dict['E_dt'][i]

    ]
    
    flattened_data.append(row)
    i = i + 1

# Convert the flattened data into a DataFrame
df = pd.DataFrame(flattened_data, columns=['G_site', 'G_lat', 'G_lon','G_alt', 'G_RHi', 'G_dt', 'E_alt', 'E_RHi', 'E_dt'], index=index)

# Now, convert the DataFrame to an xarray Dataset
dataset = xr.Dataset.from_dataframe(df)


  0%|          | 0/10 [00:00<?, ?it/s]/home/chinahg/.conda/envs/contrails/lib/python3.9/site-packages/gribapi/__init__.py:23: UserWarning: ecCodes 2.31.0 or higher is recommended. You are running version 2.30.0
  warnings.warn(
/tmp/ipykernel_92613/2495222050.py:105: DeprecationWarning: `np.float` is a deprecated alias for the builtin `float`. To silence this warning, use `float` by itself. Doing this will not modify any behavior and is safe. If you specifically wanted the numpy scalar type, use `np.float64` here.
Deprecated in NumPy 1.20; for more details and guidance: https://numpy.org/devdocs/release/1.20.0-notes.html#deprecations
  E_latitude = np.float(E_data.latitude.sel(latitude=G_lat_data[j][0], method='nearest').values)
/tmp/ipykernel_92613/2495222050.py:106: DeprecationWarning: `np.float` is a deprecated alias for the builtin `float`. To silence this warning, use `float` by itself. Doing this will not modify any behavior and is safe. If you specifically wanted the numpy scala

In [4]:
dataset

ValueError: can only convert an array of size 1 to a Python scalar

ValueError: can only convert an array of size 1 to a Python scalar

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Define the 4 points (x1, y1), (x2, y2), (x3, y3), (x4, y4)
x1, y1, z1 = 1, 1, 10  # Point 1
x2, y2, z2 = 2, 1, 20  # Point 2
x3, y3, z3 = 1, 2, 30  # Point 3
x4, y4, z4 = 2, 2, 40  # Point 4

# Define the query point (x, y) inside the rectangle formed by the 4 points
x_query = 1.1
y_query = 1.9

# Step 1: Interpolate along the x-direction (between points 1 and 2, 3 and 4)
def lerp(x0, x1, y0, y1, x):
    return y0 + (x - x0) * (y1 - y0) / (x1 - x0)

# Interpolate in the x-direction for y = 1 (between points (x1, y1) and (x2, y2))
z_left = lerp(x1, x2, z1, z2, x_query)

# Interpolate in the x-direction for y = 2 (between points (x3, y3) and (x4, y4))
z_right = lerp(x3, x4, z3, z4, x_query)

# Step 2: Interpolate in the y-direction (between the results from the previous step)
z_result = lerp(y1, y3, z_left, z_right, y_query)

# Plotting the points and the interpolation process

fig = plt.figure(figsize=(8, 6))
ax = fig.add_subplot(111)

# Plot the corner points as red markers
ax.scatter([x1, x2, x3, x4], [y1, y2, y3, y4], color='red', label='Corners')

# Annotate the corner points
ax.text(x1, y1, f'({x1}, {y1}) = {z1}', fontsize=12, verticalalignment='bottom', horizontalalignment='right')
ax.text(x2, y2, f'({x2}, {y2}) = {z2}', fontsize=12, verticalalignment='bottom', horizontalalignment='left')
ax.text(x3, y3, f'({x3}, {y3}) = {z3}', fontsize=12, verticalalignment='top', horizontalalignment='right')
ax.text(x4, y4, f'({x4}, {y4}) = {z4}', fontsize=12, verticalalignment='top', horizontalalignment='left')

# Plot the query point
ax.scatter(x_query, y_query, color='blue', label=f'Query Point ({x_query}, {y_query})')

# Draw lines connecting the query point to the interpolation steps
ax.plot([x1, x_query], [y1, y_query], 'k--', linewidth=1)
ax.plot([x2, x_query], [y2, y_query], 'k--', linewidth=1)
ax.plot([x3, x_query], [y3, y_query], 'k--', linewidth=1)
ax.plot([x4, x_query], [y4, y_query], 'k--', linewidth=1)

# Add a colorbar for the interpolation grid (optional)
grid_x = np.linspace(1, 2, 100)
grid_y = np.linspace(1, 2, 100)
grid_X, grid_Y = np.meshgrid(grid_x, grid_y)

# Create a grid of interpolated values
Z = np.array([[lerp(x1, x2, lerp(y1, y3, z1, z3, y), lerp(y2, y4, z2, z4, y), x) 
              for x in grid_x] for y in grid_y])

# Plot the interpolation as a heatmap
c = ax.pcolormesh(grid_X, grid_Y, Z, shading='auto', cmap='coolwarm', alpha=0.5)
fig.colorbar(c, ax=ax, label='Interpolated Value')

# Labels and title
ax.set_xlabel('X')
ax.set_ylabel('Y')
ax.set_title('2D Linear Interpolation')

# Show the plot with the query point and the interpolation heatmap
ax.legend()
plt.show()

# Output interpolated result
print(f"The interpolated value at ({x_query}, {y_query}) is {z_result}")
